In [3]:
import os
import pickle
import numpy as np
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model

In [4]:
IMG_SIZE = (128, 128)
ENCODER_PATH = "face_encoder.keras"
GALLERY_DIR = "gallery"
GALLERY_CACHE = "gallery_embeddings.pkl"
DISTANCE_THRESHOLD = 0.9

In [7]:
class L2Normalize(keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

    def compute_output_shape(self, input_shape):
        return input_shape


class L1Distance(keras.layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return tf.abs(a - b)

    def compute_output_shape(self, input_shape):
        return input_shape[0]


encoder = load_model(
    ENCODER_PATH,
    custom_objects={"L2Normalize": L2Normalize, "L1Distance": L1Distance},
)
print("Encoder loaded. Output shape:", encoder.output_shape)

Encoder loaded. Output shape: (None, 128)


In [8]:
def preprocess_face(face_bgr):
    face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    face_rgb = cv2.resize(face_rgb, IMG_SIZE)
    face_rgb = face_rgb.astype(np.float32) / 255.0
    return face_rgb

def get_embedding(face_bgr):
    x = preprocess_face(face_bgr)
    x = np.expand_dims(x, axis=0)
    emb = encoder.predict(x, verbose=0)[0]
    return emb

In [11]:
def build_gallery(gallery_dir):
    gallery = {}

    if not os.path.exists(gallery_dir):
        os.makedirs(gallery_dir, exist_ok=True)
        print(f"Folder '{gallery_dir}' did not exist and was created.")
        print(f"Now, inside '{os.path.abspath(gallery_dir)}', create one subfolder per person (named after them) and put their photos inside it, then re-run this cell.")
        return gallery

    person_names = [
        p for p in os.listdir(gallery_dir)
        if os.path.isdir(os.path.join(gallery_dir, p))
    ]

    if not person_names:
        print(f"Folder '{gallery_dir}' is empty. Create one subfolder per person and add their photos.")
        return gallery

    for person_name in person_names:
        person_dir = os.path.join(gallery_dir, person_name)
        embeddings = []
        for fname in os.listdir(person_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            img_path = os.path.join(person_dir, fname)
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: could not read image -> {img_path}")
                continue
            embeddings.append(get_embedding(img))
        if embeddings:
            gallery[person_name] = embeddings
            print(f"{person_name}: {len(embeddings)} images processed")
        else:
            print(f"Warning: no valid images found for '{person_name}'.")

    return gallery


print("Current working directory:", os.getcwd())
print("Gallery path:", os.path.abspath(GALLERY_DIR))

if os.path.exists(GALLERY_CACHE):
    with open(GALLERY_CACHE, "rb") as f:
        gallery = pickle.load(f)
    print("Gallery loaded from cache.")
else:
    gallery = build_gallery(GALLERY_DIR)
    if gallery:
        with open(GALLERY_CACHE, "wb") as f:
            pickle.dump(gallery, f)
        print("Gallery built and saved.")
    else:
        print("Gallery is empty — add images before continuing, then re-run this cell.")

print("People in gallery:", list(gallery.keys()))

Current working directory: C:\Users\Acer\deep
Gallery path: C:\Users\Acer\deep\gallery
sogol: 4 images processed
Gallery built and saved.
People in gallery: ['sogol']


In [12]:
def recognize(embedding, gallery, threshold=DISTANCE_THRESHOLD):
    best_name = "Unknown"
    best_dist = float("inf")
    for name, embeddings in gallery.items():
        for gal_emb in embeddings:
            dist = np.linalg.norm(embedding - gal_emb)
            if dist < best_dist:
                best_dist = dist
                best_name = name
    if best_dist > threshold:
        return "Unknown", best_dist
    return best_name, best_dist

In [ ]:
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("erroe.")

print("click on.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))

    for (x, y, w, h) in faces:
        face_crop = frame[y:y + h, x:x + w]
        embedding = get_embedding(face_crop)
        name, dist = recognize(embedding, gallery)

        color = (0, 200, 0) if name != "Unknown" else (0, 0, 200)
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
        label = f"{name} ({dist:.2f})"
        cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    cv2.imshow("Face Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()